# Set Up Notebook

In [ ]:
##########
# IMPORT #
##########

# Data processing and math
import pandas as pd
import numpy as np

# Statistics
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.formula.api import ols

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Display handling
import warnings


#################
# CONFIGURATION #
#################

# Suppress warnings
warnings.filterwarnings('ignore')

# Set global plotting style
sns.set_theme(style = "whitegrid")


##########
# LABELS #
##########
labels_gender = {
    '1': 'Female',
    '2': 'Male',
    '3': 'Non-binary',
    '4': 'Prefer not to say'
}


#################
# VISUALIZATION #
#################
palette_main = {"Non-Ignorant": "#0072B2", "Ignorant": "#D55E00"}

# Transform Data

In [ ]:
# Load data
df_clean = pd.read_csv('blame_praise_self_study_3_data.csv', sep=';')

# Define factors
factors_map = {
    'non,blameAc':         {'Upbringing': 'Non-Ignorant', 'Action': 'Harm (Blame)',  'Measure': 'Blame / Praise'},
    'non,blameSelf':       {'Upbringing': 'Non-Ignorant', 'Action': 'Harm (Blame)',  'Measure': 'True Self'},
    'non,praiseAc':        {'Upbringing': 'Non-Ignorant', 'Action': 'Help (Praise)', 'Measure': 'Blame / Praise'},
    'non,praiseSelf':      {'Upbringing': 'Non-Ignorant', 'Action': 'Help (Praise)', 'Measure': 'True Self'},
    'ignorant,blame,AC':   {'Upbringing': 'Ignorant',     'Action': 'Harm (Blame)',  'Measure': 'Blame / Praise'},
    'ignorant,blameSelf':  {'Upbringing': 'Ignorant',     'Action': 'Harm (Blame)',  'Measure': 'True Self'},
    'ignorant,praiseAc':   {'Upbringing': 'Ignorant',     'Action': 'Help (Praise)', 'Measure': 'Blame / Praise'},
    'ignorant,praiseSelf': {'Upbringing': 'Ignorant',     'Action': 'Help (Praise)', 'Measure': 'True Self'}
}

def extract_score(val):
    if pd.isna(val): return np.nan
    s = str(val)[0]
    return int(s) if s.isdigit() else np.nan

# Reshape wide to long
long_data = []
for _, row in df_clean.iterrows():
    for col, factors in factors_map.items():
        score = extract_score(row.get(col))
        if not np.isnan(score):
            entry = {'ID': row['ID'], 'Score': score}
            entry.update(factors)
            long_data.append(entry)

df_long = pd.DataFrame(long_data)

print(f"Transformation complete ({len(df_long)} Observations).")

# Sociodemographics

In [ ]:
df_clean['Gender_Labeled'] = df_clean['Gender'].astype(str).map(labels_gender)
df_clean['Age'] = pd.to_numeric(df_clean['Age'], errors = 'coerce')

for col in ['Gender_Labeled']:
    print(f"\n{col}:")
    display(df_clean[col].value_counts().to_frame('Frequency'))

print("\nAge:")
display(df_clean['Age'].describe().to_frame('Value').round(3))

# Calculate Descriptive Statistics

In [ ]:
# Calculate descriptive statistics
descriptive_stats = df_long.groupby(['Measure', 'Action', 'Upbringing'])['Score'].agg(['mean', 'std', 'count']).round(3)

# Display results
display(descriptive_stats)

# Perform ANOVAs

In [ ]:
for measure in df_long['Measure'].unique():
    print(f"\n2 × 2 ANOVA ({measure})")
    subset = df_long[df_long['Measure'] == measure]

    # Fit model
    model = ols('Score ~ C(Upbringing) * C(Action)', data = subset).fit()
    
    # Run ANOVA
    aov_table = sm.stats.anova_lm(model, typ = 2)
    
    # Calculate effect sizes
    aov_table['partial_eta_sq'] = aov_table['sum_sq'] / (aov_table['sum_sq'] + aov_table.loc['Residual', 'sum_sq'])
    
    # Display results
    display(aov_table.round(3))

# Perform t-Tests

In [ ]:
t_results = []

for measure in df_long['Measure'].unique():
    for action in df_long['Action'].unique():

        # Subset data for comparing Non-Ignorant vs. Ignorant
        norm = df_long[(df_long['Measure'] == measure) & (df_long['Action'] == action) & (df_long['Upbringing'] == 'Non-Ignorant')]['Score']
        isol = df_long[(df_long['Measure'] == measure) & (df_long['Action'] == action) & (df_long['Upbringing'] == 'Ignorant')]['Score']

        # Perform Welch's t-test
        t_stat, p_val = stats.ttest_ind(norm, isol, equal_var = False)
        
        # Store results in list
        t_results.append({
            'DV': measure, 
            'Condition': action,
            'Mean Non-Ignorant': round(norm.mean(), 3), 
            'Mean Ignorant': round(isol.mean(), 3),
            't': round(t_stat, 3), 
            'p': round(p_val, 3)
        })

# Display results
display(pd.DataFrame(t_results))

# Generate Bar Plots

In [ ]:
for measure in df_long['Measure'].unique():
    plt.figure(figsize = (8, 5))

    # Create graph
    g = sns.barplot(data = df_long[df_long['Measure'] == measure],
                    x = "Action",
                    y = "Score",
                    hue = "Upbringing",
                    palette = palette_main,
                    capsize = .05,
                    errorbar = "se"
                   )
    
    # Set axis labels and titles
    plt.ylabel('Mean Score (1 to 7)', fontsize = 12)
    plt.xlabel('Action', fontsize = 12)
    plt.ylim(1, 7.5)
    
    # Add main title
    plt.title(f'{measure}', fontsize = 14)
    
    # Add legend
    plt.legend(title = 'Upbringing', bbox_to_anchor = (1.05, 1), loc = 'upper left')
    
    # Export graph
    clean_name = measure.replace("/", "or").replace(" ", "_").lower()
    filename = f'blame_praise_self_study_3_{clean_name}.png'
    plt.savefig(filename, dpi = 300, bbox_inches = 'tight')
    plt.show()
    print(f"Graph saved as '{filename}'.")